# Rivalry Pulse preprocessing
This notebook orchestrates the selective Olympedia pipeline implemented in `rivalry_scraper.py`. Download and parsing are deliberately separate so parsing can be rerun offline.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import rivalry_scraper as rs


In [ ]:
ROOT = Path('.')
INPUT = ROOT.parent / 'olympics' / 'Olympic_Athlete_Event_Results.csv'
CACHE = ROOT / 'cache'
OUTPUT = ROOT / 'output'
OUTPUT.mkdir(exist_ok=True)


## 1. Inspect the local source
No network access is used.


In [ ]:
source = rs.load_source_csv(INPUT)
analysis = rs.analyze_source(source)
analysis


## 2. Generate selective candidate result IDs
A candidate is only an event where USA and URS both appear **and** the sport/event can contain a literal binary encounter. Co-presence is not treated as a match.


In [ ]:
candidates = rs.build_candidates(source)
candidates.to_csv(OUTPUT / 'rivalry_pulse_candidates.csv', index=False)
candidates.groupby('sport').size().sort_values(ascending=False)


## 3. One-time acquisition (run intentionally)
For server safety the recommended live acquisition is the CLI `download` command documented in README. It is sequential, cache-first and refuses delays below 4 seconds. Keep this notebook offline by default.

Example command:
```bash
python rivalry_scraper.py download --candidates output/rivalry_pulse_candidates.csv --cache-dir cache --manifest output/cache_manifest.csv --stats-output output/download_stats.json --delay 4.5 --user-agent "Olympic-Cold-War-DataViz/1.0 (University project; contact: YOUR_CONTACT)"
```


## 4. Offline parsing and validation
This section reads only files already present in `cache/`.


In [ ]:
matches, page_audit, parse_stats, issues = rs.parse_cached_pages(candidates, source, CACHE)
matches.to_csv(OUTPUT / 'rivalry_pulse_matches.csv', index=False)
page_audit.to_csv(OUTPUT / 'candidate_page_audit.csv', index=False)
pd.DataFrame(issues).to_csv(OUTPUT / 'validation_issues.csv', index=False)
parse_stats


In [ ]:
download_stats = rs.read_download_stats(OUTPUT / 'download_stats.json')
report = rs.build_validation_report(analysis, candidates, matches, page_audit, parse_stats, issues, download_stats)
rs.write_json(OUTPUT / 'validation_report.json', report)
report['outcomes'], report['matches_by_edition'], report['matches_by_sport']


## 5. Visualization-ready subset
`counts_for_pulse` excludes unplayed pairings such as byes/walkovers while retaining them in the auditable raw match output.


In [ ]:
pulse = matches[matches['counts_for_pulse'].astype(bool)].copy() if not matches.empty else matches.copy()
pulse.head()
